In [1]:
import sys

class BufferOverflowSimulator:
    def __init__(self):
        # Simulated Memory Layout (Stack frame representation)
        self.buffer_size = 10
        self.reset_memory()

    def reset_memory(self):
        # 10 bytes for buffer, followed by adjacent 4-byte variable `authenticated`
        self.buffer = [b'\x00'] * self.buffer_size
        self.authenticated = 0  # 0 = Access Denied, 1 = Granted (4-byte int: 0x00000000)

    def display_memory(self, title):
        print(f"\n==================================================")
        print(f" {title}")
        print(f"==================================================")
        print(f"[Memory State]")
        
        # Format buffer display
        buf_repr = "".join([c.decode('latin-1') if c != b'\x00' else '.' for c in self.buffer])
        buf_hex = " ".join([f"{ord(c):02X}" for c in self.buffer])
        
        print(f"  Buffer (Size 10)     : [{buf_repr}] | Hex: [{buf_hex}]")
        print(f"  `authenticated` Var : {self.authenticated} (Hex: 0x{self.authenticated:08X})")
        
        # Access control status
        if self.authenticated != 0:
            print("  --> [!] ACCESS GRANTED (Memory Overwritten!)")
        else:
            print("  --> [ ] ACCESS DENIED (Normal Operation)")
        print("--------------------------------------------------")

    def run_simulation(self):
        while True:
            self.reset_memory()
            self.display_memory("PROGRAM TITLE: Legacy C Authentication Module (Unchecked Input)")

            print("\n[Input]")
            print("  Target Buffer Length : 10 bytes")
            print("  Adjacent Control Var : `authenticated` (starts at 0)")
            print("  Tip: Enter >10 characters to cause an overflow (e.g., 'AAAAAAAAAAAA\x01').")
            
            try:
                user_input = input("\nEnter input string: ")
            except (KeyboardInterrupt, EOFError):
                print("\n\nExiting program...")
                sys.exit(0)

            # Convert input string to bytes
            input_bytes = [bytes([b]) for b in user_input.encode('latin-1', errors='replace')]

            # Vulnerable C behavior simulation (Unchecked memcpy/strcpy)
            # Unchecked loop writes beyond buffer array boundary into `authenticated` variable
            for i, char_byte in enumerate(input_bytes):
                if i < self.buffer_size:
                    self.buffer[i] = char_byte
                elif i < self.buffer_size + 4:
                    # Overwriting adjacent 4-byte integer memory (Little-Endians byte ordering)
                    shift = (i - self.buffer_size) * 8
                    byte_val = ord(char_byte)
                    
                    # Clear byte mask and overwrite
                    mask = ~(0xFF << shift)
                    self.authenticated = (self.authenticated & mask) | (byte_val << shift)

            # Display updated memory output
            self.display_memory("OUTPUT & MEMORY ANALYZER")

            # Exception / Security Status Handling
            if len(input_bytes) > self.buffer_size:
                print("\n[SECURITY EXCEPTION HANDLER]")
                print("  STATUS   : BUFFER OVERFLOW DETECTED")
                print(f"  DETAILS  : Input of length {len(input_bytes)} exceeded allocation of {self.buffer_size} bytes.")
                if self.authenticated != 0:
                    print("  CRITICAL : Memory boundary violation corrupted `authenticated` flag!")
            else:
                print("\n[SECURITY STATUS]")
                print("  STATUS   : NORMAL EXECUTION")
                print("  DETAILS  : Input within safe boundaries.")

            # Continue or Exit option loop
            print("\n--------------------------------------------------")
            while True:
                choice = input("Do you want to continue simulation? (c = continue / e = exit): ").strip().lower()
                if choice in ['c', 'continue']:
                    break
                elif choice in ['e', 'exit']:
                    print("\nTerminating simulation. Goodbye!")
                    sys.exit(0)
                else:
                    print("Invalid option. Please enter 'c' to continue or 'e' to exit.")

if __name__ == "__main__":
    sim = BufferOverflowSimulator()
    sim.run_simulation()


 PROGRAM TITLE: Legacy C Authentication Module (Unchecked Input)
[Memory State]
  Buffer (Size 10)     : [..........] | Hex: [00 00 00 00 00 00 00 00 00 00]
  `authenticated` Var : 0 (Hex: 0x00000000)
  --> [ ] ACCESS DENIED (Normal Operation)
--------------------------------------------------

[Input]
  Target Buffer Length : 10 bytes
  Adjacent Control Var : `authenticated` (starts at 0)
  Tip: Enter >10 characters to cause an overflow (e.g., 'AAAAAAAAAAAA').



Enter input string:  aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaaa



 OUTPUT & MEMORY ANALYZER
[Memory State]
  Buffer (Size 10)     : [aaaaaaaaaa] | Hex: [61 61 61 61 61 61 61 61 61 61]
  `authenticated` Var : 1633771873 (Hex: 0x61616161)
  --> [!] ACCESS GRANTED (Memory Overwritten!)
--------------------------------------------------

[SECURITY EXCEPTION HANDLER]
  STATUS   : BUFFER OVERFLOW DETECTED
  DETAILS  : Input of length 41 exceeded allocation of 10 bytes.
  CRITICAL : Memory boundary violation corrupted `authenticated` flag!

--------------------------------------------------


Do you want to continue simulation? (c = continue / e = exit):  e



Terminating simulation. Goodbye!


SystemExit: 0

C:\Users\26dpscy0001\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\IPython\core\interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
